# Day 15 — DataFrame Fundamentals & Labeled Data Structures

This notebook covers Series, DataFrames, Memory Layout, and Selection Mechanics (`.loc` vs `.iloc`)

## Validation goals
- Understand how Pandas `Series` and `DataFrame` structures align with underlying NumPy memory blocks[cite: 10]
- Construct DataFrames using dictionaries, `pd.date_range()`, and explicit index labeling[cite: 10]
- Analyze internal memory overhead using `df.info()` and `df.memory_usage(deep=True)`[cite: 10]
- Master precise operational distinctions between label-based indexing (`.loc`) and positional indexing (`.iloc`)[cite: 10]
- Perform vectorized calculations across DataFrame columns using `.to_numpy()` and native Series arithmetic[cite: 10]

Pandas builds directly on top of NumPy array blocks. While a NumPy array is a contiguous, homogeneous memory block, a Pandas DataFrame is a collection of 1D labeled arrays (`Series`), where each column can hold a different data type (`float64`, `int64`, `object`, `datetime64`).

Understanding how Pandas maps index labels to internal NumPy memory locations is critical for writing efficient, vectorized tabular operations.

Core Concepts
1. Series vs. DataFrame
Series: A 1D labeled array wrapping a 1D NumPy ndarray alongside an explicit index.

DataFrame: A 2D tabular data structure with aligned row labels (index) and column labels (columns). Under the hood, Pandas groups columns of the same dtype into Block Managers wrapping NumPy multidimensional arrays.

2. Explicit vs. Implicit Indexing
`.loc[]:` Label-based indexing. Uses explicit index/column names. Slicing with `.loc` is inclusive of both start and end bounds (`loc['a':'c'] includes 'c'`).

`.iloc[]:` Positional indexing. Uses 0-indexed integer offsets (identical to standard NumPy slicing). Slicing with .iloc is exclusive of the upper bound (`iloc[0:3]` includes indices 0, 1, 2).

3. Memory Layout & dtypes
Inspecting `df.info()` and `df.memory_usage(deep=True)` reveals how string objects (`object` or `string dtype`) consume significantly more pointer overhead than numeric primitives (`float64`, `int64`). Vectorized operations leverage underlying continuous contiguous memory blocks.

### Hands-On Implementation:

Here is a practical script introducing Series creation, DataFrame construction from dictionaries and NumPy arrays, memory inspection, and the precise operational differences between `.loc` and `.iloc`.

In [1]:
import numpy as np
import pandas as pd

# 1. Creating Series and DataFrame from NumPy Arrays & Dictionaries
np.random.seed(42)

# Simulating 5 days of multi-sensor telemetry readings
timestamps = pd.date_range(start="2026-10-18", periods=5, freq="D")
sensor_ids = [f"SENSOR_{i:03d}" for i in range(101, 106)]

# Heterogeneous dataset
data = {
    "timestamp": timestamps,
    "sensor_id": sensor_ids,
    "voltage": np.round(np.random.uniform(3.1, 4.2, size=5), 2),
    "current_ma": np.random.randint(150, 500, size=5),
    "is_active": [True, True, False, True, True],
}

df = pd.DataFrame(data)

# Set custom explicit index
df.set_index("sensor_id", inplace=False)
df_indexed = df.set_index("sensor_id")

print("--- DataFrame Structure ---")
print(df_indexed)
print("\n--- Memory Inspection ---")
print(df_indexed.info())
print("\nExact Memory Usage (Bytes):")
print(df_indexed.memory_usage(deep=True))

# 2. Precise Indexing: .loc vs .iloc
print("\n--- Label-based Selection (.loc) ---")
# Select specific rows by explicit index label and columns by name
loc_selection = df_indexed.loc["SENSOR_102":"SENSOR_104", ["voltage", "is_active"]]
print(loc_selection)

print("\n--- Positional Selection (.iloc) ---")
# Select rows 1 to 3 (exclusive) and columns 0 to 2
iloc_selection = df_indexed.iloc[1:3, 0:2]
print(iloc_selection)

# 3. Vectorized Column Assignment (NumPy Integration)
# Convert Series to underlying NumPy array directly via .values or .to_numpy()
raw_voltage_array = df_indexed["voltage"].to_numpy()
df_indexed["power_mw"] = df_indexed["voltage"] * df_indexed["current_ma"]

print("\n--- Extended DataFrame with Vectorized Math ---")
print(df_indexed[["voltage", "current_ma", "power_mw"]])

--- DataFrame Structure ---
            timestamp  voltage  current_ma  is_active
sensor_id                                            
SENSOR_101 2026-10-18     3.51         364       True
SENSOR_102 2026-10-19     4.15         480       True
SENSOR_103 2026-10-20     3.91         237      False
SENSOR_104 2026-10-21     3.76         249       True
SENSOR_105 2026-10-22     3.27         301       True

--- Memory Inspection ---
<class 'pandas.DataFrame'>
Index: 5 entries, SENSOR_101 to SENSOR_105
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   timestamp   5 non-null      datetime64[us]
 1   voltage     5 non-null      float64       
 2   current_ma  5 non-null      int32         
 3   is_active   5 non-null      bool          
dtypes: bool(1), datetime64[us](1), float64(1), int32(1)
memory usage: 145.0+ bytes
None

Exact Memory Usage (Bytes):
Index         295
timestamp      40
voltage        40
curre